<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-06-reliable-structured-extraction-for-meridian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6 (graded) — Reliable structured extraction for Meridian
**Course 2: Generative AI and LLMs with Python — Chapter 6: Prompting & structured output**

**Problem brief (Leo Farkas, Meridian Bank):** "We need specific fields pulled out of
filings — as clean JSON, every time, no prose." Target: schema-valid JSON on ≥ 99% of
documents, a field-level accuracy target.

**What you'll submit:** a working extractor with schema validation, a 20+ item eval set, a
repair loop for failures, and a scorecard reporting schema validity + field accuracy.

In [ ]:
!pip install -q jsonschema transformers

## 1. Data: filing excerpts with known ground truth (for grading)
Real named-entity SEC filing text is highly variable; this lab uses a template-generated set
of realistic filing excerpts **with known ground truth**, so field-level accuracy is actually
measurable — the same approach a real evaluation set needs (label the field you're checking).

In [ ]:
import random
random.seed(0)

names = ['Meridian Capital LLC', 'Northgate Holdings Inc.', 'Riverside Partners LP',
         'Alden Manufacturing Co.', 'Bluepeak Ventures Ltd.', 'Cascade Retail Group']

def make_filing(i):
    name = random.choice(names)
    amount = random.choice([250000, 1_500_000, 4_200_000, 750000, 12_000_000])
    year = random.choice([2022, 2023, 2024])
    month, day = random.randint(1, 12), random.randint(1, 28)
    date = f'{year:04d}-{month:02d}-{day:02d}'
    text = (
        f"On {date}, {name} (the 'Borrower') entered into a credit facility with the Bank "
        f"in the principal amount of ${amount:,}. The facility is secured by the Borrower's "
        f"receivables and is subject to customary covenants set forth in Exhibit A."
    )
    return {'id': i, 'text': text,
            'ground_truth': {'borrower_name': name, 'loan_amount': amount, 'filing_date': date}}

eval_set = [make_filing(i) for i in range(25)]
print(eval_set[0]['text'])
print(eval_set[0]['ground_truth'])

## 2. The schema + a rule-based extractor as the offline fallback
The offline path uses careful regex, not an LLM — it always produces schema-valid output and
is what the notebook falls back to if no model call succeeds, matching the zero-cost
guarantee. The real exercise is the LLM path below it, with the repair loop.

In [ ]:
import re
import json
from jsonschema import validate, ValidationError

SCHEMA = {
    'type': 'object',
    'properties': {
        'borrower_name': {'type': 'string'},
        'loan_amount': {'type': 'number'},
        'filing_date': {'type': 'string', 'pattern': r'^\d{4}-\d{2}-\d{2}$'},
    },
    'required': ['borrower_name', 'loan_amount', 'filing_date'],
}

def rule_based_extract(text):
    date_m = re.search(r'On (\d{4}-\d{2}-\d{2})', text)
    name_m = re.search(r', (.+?) \(the', text)
    amount_m = re.search(r'\$([\d,]+)', text)
    return {
        'borrower_name': name_m.group(1) if name_m else '',
        'loan_amount': int(amount_m.group(1).replace(',', '')) if amount_m else 0,
        'filing_date': date_m.group(1) if date_m else '',
    }

## 3. The LLM path, with a retry/repair loop

In [ ]:
import os
try:
    from google.colab import userdata
    API_KEY = userdata.get('LLM_API_KEY'); BASE_URL = userdata.get('LLM_BASE_URL')
except Exception:
    API_KEY = os.environ.get('LLM_API_KEY'); BASE_URL = os.environ.get('LLM_BASE_URL')
hosted_available = bool(API_KEY and BASE_URL)

EXTRACTION_PROMPT = (
    'Extract these three fields from the filing text as JSON ONLY, no prose, matching this '
    'schema exactly: {"borrower_name": string, "loan_amount": number (no $ or commas), '
    '"filing_date": "YYYY-MM-DD"}.\n\nText: {text}\n\nJSON:'
)

def call_llm(text):
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
    resp = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': EXTRACTION_PROMPT.format(text=text)}],
        max_tokens=100, temperature=0.0,
        response_format={'type': 'json_object'},  # JSON mode, where the provider supports it
    )
    return resp.choices[0].message.content

def extract_with_repair(text, max_retries=2):
    """Try the LLM (if configured), validate against SCHEMA, retry with the error fed back
    in the prompt on failure, and fall back to the rule-based extractor if every attempt
    fails or no hosted model is configured."""
    if not hosted_available:
        return rule_based_extract(text), 'offline-fallback'

    last_error = None
    for attempt in range(max_retries + 1):
        try:
            raw = call_llm(text)
            parsed = json.loads(raw)
            validate(instance=parsed, schema=SCHEMA)
            return parsed, f'llm-attempt-{attempt}'
        except (json.JSONDecodeError, ValidationError) as e:
            last_error = e
            continue  # a real repair loop would feed `last_error` back into the next prompt
    print(f'All LLM attempts failed ({last_error}) — falling back to the rule-based extractor.')
    return rule_based_extract(text), 'offline-fallback-after-llm-failure'

## 4. Run the eval set and score it

In [ ]:
results = []
for item in eval_set:
    extracted, source = extract_with_repair(item['text'])
    schema_valid = True
    try:
        validate(instance=extracted, schema=SCHEMA)
    except ValidationError:
        schema_valid = False
    gt = item['ground_truth']
    field_correct = {
        f: (extracted.get(f) == gt[f]) for f in gt
    }
    results.append({'id': item['id'], 'source': source, 'schema_valid': schema_valid, **field_correct})

import pandas as pd
results_df = pd.DataFrame(results)

schema_validity_rate = results_df['schema_valid'].mean()
field_accuracy = {f: results_df[f].mean() for f in ['borrower_name', 'loan_amount', 'filing_date']}

print(f'Schema validity rate: {schema_validity_rate:.1%}  (target: >= 99%)')
for f, acc in field_accuracy.items():
    print(f'  {f:15s} accuracy: {acc:.1%}')
results_df.head(10)

## 5. Scorecard write-up (fill in)
If you had no hosted API key, every row used the offline rule-based extractor — note that
explicitly and explain why it's a legitimate stand-in for grading purposes (fixed-format
text, deterministic, always schema-valid) but would NOT generalize to real, messier filing
text the way an LLM extractor is meant to. If you did have a key, which fields were hardest
for the model, and why?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 6: Prompting & structured output*